<a href="https://colab.research.google.com/github/kittiporntum1-cell/ADALL_github/blob/main/Tum_Co_pilot_guidance__Copy_of_ADALL_Practical_Test_Revision_Regression_vold2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Practical Test Revision (Regression): Predicting Number of Failed Subjects

This notebook revises a **regression workflow** using a target that is a **special case**:

- Your target is a **count**: number of subjects failed (based on scores below **10** from **G1, G2, G3**).
- Because the target is an **integer count** (often 0, 1, 2, 3), it can *feel* similar to classification.
- However, you are still doing **regression** because the model outputs a **numeric value** and you evaluate using **regression metrics**.

## How to think about this target (important for revision)

| Perspective | What it means here | Typical choice |
|---|---|---|
| Regression view | Predict a numeric count (can be 0–3, but model can output non-integers) | MSE / MAE / RMSE, R² |
| Classification-like view (edge case) | If you convert counts into categories (e.g., 0 vs ≥1) you can classify | Accuracy, F1, ROC-AUC |

### Caveat you must remember
Even when the target behaves like classes, **do not switch to classification** unless the task explicitly asks you to do so.

## Practical test habits
- Keep code **readable** and **step-by-step**.
- Add short notes explaining **why** a step is needed.
- When unsure: print shapes, check missing values, and check your target distribution.

---


STEP 0  Setup

STEP 1  Load Data

STEP 2  Explore Dataset
         df.shape
         df.info()
         df.describe()
         df.isnull().sum()

STEP 3  Dataset Profile
    dataset_profile

Markdown:
    - What the dataset is about
    - Target definition
    - Dataset imbalance
    - Evaluation metric choice

STEP 3A LLM Dataset Reasoning Prompt

STEP 4  LLM assisted problem framing

STEP 5  Context

STEP 6  Instruction

STEP 7  Payload

STEP 8  OpenAI Recommendations

STEP 9  Review Recommendations

STEP 10  Data Cleaning
         - Missing values
         - Duplicates
         - Invalid values
         - Data types
         - Remove leakage columns (if instructed)

STEP 11 Create Target
          y = target

STEP 12 Jitter Plots (Predictor vs Target Analysis)


STEP 13  Train/Test Split

STEP 14 Preprocessing
          - Encoding
          - Scaling
          - ColumnTransformer
          - Pipeline

STEP 15 Baseline Model

STEP 16 Evaluation
          - MAE
          - RMSE
          - R²

STEP 17 Improvements
          - Hyperparameter tuning
          - Feature engineering
          - Compare models

## **Step 0) Setup**

In [17]:
# Core libraries

import os
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Modelling libraries used later in Session 2
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Make wide tables easier to read in Colab
pd.set_option('display.max_columns', 100)

# **Step 0A: OpenAI Setup**

In [18]:
from google.colab import userdata
from openai import OpenAI

# load key from Colab Secrets
api_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=api_key
)
print("OpenAI client created successfully.")

OpenAI client created successfully.



#**#Step 1) Load data and do quick checks**

**Goal:** confirm you loaded the correct file, and the dataset looks sensible.

Checklist:
- Use `df.head()` to sanity check columns.
- Use `df.shape` to confirm rows and columns.
- Use `df.info()` to see dtypes and missing values quickly.

If anything looks odd (unexpected columns, too few rows), stop and fix it before modelling.


In [ ]:
# another load dataset template (github)

github_raw_url = 'https://raw.githubusercontent.com/rq-goh/ADALL_github/refs/heads/main/laptop_prices_2024_sgd_TL.csv'

df = pd.read_csv(github_raw_url)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

In [19]:
# ------------------------------------------------------------
# LOAD DATA (do first) , Template for Kagglehub data set
# Tip: if this block is slow, run it early.
# ------------------------------------------------------------

import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("devansodariya/student-performance-data")
print("Downloaded to:", path)
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "student_data.csv"))
df.head()

Using Colab cache for faster access to the 'student-performance-data' dataset.
Downloaded to: /kaggle/input/student-performance-data
['student_data.csv']


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10


In [ ]:
# another load dataset template (csv file)
df = pd.read_csv(...)
print(df.shape)
display(df.head())

# **Step 2: Understand Dataset**

In [20]:
df = pd.read_csv(os.path.join(path, "student_data.csv"))
df.shape # Number of rows & columns

(395, 33)

In [21]:
df.info() #dataset structure

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

In [22]:
df.describe(include='all')

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,395,395,395.000000,395,395,395,395.000000,395.000000,395,395,395,395,395.000000,395.000000,395.000000,395,395,395,395,395,395,395,395,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
unique,2,2,NaN,2,2,2,NaN,NaN,5,5,4,3,NaN,NaN,NaN,2,2,2,2,2,2,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,GP,F,NaN,U,GT3,T,NaN,NaN,other,other,course,mother,NaN,NaN,NaN,no,yes,no,yes,yes,yes,yes,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,349,208,NaN,307,281,354,NaN,NaN,141,217,145,273,NaN,NaN,NaN,344,242,214,201,314,375,329,263,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,16.696203,NaN,NaN,NaN,2.749367,2.521519,NaN,NaN,NaN,NaN,1.448101,2.035443,0.334177,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.944304,3.235443,3.108861,1.481013,2.291139,3.554430,5.708861,10.908861,10.713924,10.415190
std,NaN,NaN,1.276043,NaN,NaN,NaN,1.094735,1.088201,NaN,NaN,NaN,NaN,0.697505,0.839240,0.743651,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.896659,0.998862,1.113278,0.890741,1.287897,1.390303,8.003096,3.319195,3.761505,4.581443
min,NaN,NaN,15.000000,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,1.000000,1.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,3.000000,0.000000,0.000000
25%,NaN,NaN,16.000000,NaN,NaN,NaN,2.000000,2.000000,NaN,NaN,NaN,NaN,1.000000,1.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,8.000000,9.000000,8.000000
50%,NaN,NaN,17.000000,NaN,NaN,NaN,3.000000,2.000000,NaN,NaN,NaN,NaN,1.000000,2.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000,11.000000,11.000000,11.000000
75%,NaN,NaN,18.000000,NaN,NaN,NaN,4.000000,3.000000,NaN,NaN,NaN,NaN,2.000000,2.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,13.000000,13.000000,14.000000


In [23]:
df.isnull().sum()

,0
school,0
sex,0
age,0
address,0
famsize,0
Pstatus,0
Medu,0
Fedu,0
Mjob,0
Fjob,0


In [24]:
df.columns.tolist()

['school',
 'sex',
 'age',
 'address',
 'famsize',
 'Pstatus',
 'Medu',
 'Fedu',
 'Mjob',
 'Fjob',
 'reason',
 'guardian',
 'traveltime',
 'studytime',
 'failures',
 'schoolsup',
 'famsup',
 'paid',
 'activities',
 'nursery',
 'higher',
 'internet',
 'romantic',
 'famrel',
 'freetime',
 'goout',
 'Dalc',
 'Walc',
 'health',
 'absences',
 'G1',
 'G2',
 'G3']

### What you should write in markdown as you go

In a practical test, you are often graded on your reasoning, not just your code.

As you proceed, add short notes like:
- What does each key column represent in plain words?
- What is your target, and why is it defined this way?
- Is the dataset imbalanced? If yes, what is the impact on evaluation?
- What metric did you choose, and why?

Keep each explanation to **2 to 5 lines**.


###Info about the dataset:

Student Performance Data was obtained in a survey of students' math course in secondary school.
It consists of 33 Column
Dataset Contains Features like:
```
school ID
gender
age
size of family
Father education
Mother education
Occupation of Father and Mother
Family Relation
Health
Grades
```

# **Step 3: Create Dataset Profile**  (Template to be re-used)

In [25]:
dataset_profile = f"""
Dataset Shape:
{df.shape}

Columns:
{list(df.columns)}

Data Types:
{df.dtypes.to_string()}

Missing Values:
{df.isnull().sum().to_string()}

Duplicate Rows:
{df.duplicated().sum()}

Sample Records:
{df.head().to_string()}
"""

print(dataset_profile)


Dataset Shape:
(395, 33)

Columns:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']

Data Types:
school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences      

# **Short Text (Use in practical test)**: Explanation: A text-based dataset profile was created to summarise the dataset structure, size, columns, data types, missing values, duplicate records and sample observations. This provides a concise overview of the dataset before machine learning modelling.


In [10]:
# Mr.Zaw payload template ####
# Build payload text step by step.
# Payload text is a short profile of the dataset.
# It is safer and smaller than sending the full dataset to the LLM.

payload_text = ''

# 1. Shape
payload_text += '=== SHAPE ===\n'
payload_text += 'Rows: ' + str(df.shape[0]) + '\n'
payload_text += 'Columns: ' + str(df.shape[1]) + '\n\n'

# 2. Column names and data types
payload_text += '=== COLUMNS AND DATA TYPES ===\n'
payload_text += df.dtypes.to_string()
payload_text += '\n\n'

# 3. Numeric summary
payload_text += '=== NUMERIC SUMMARY ===\n'
numeric_summary = df.describe(include='number').round(2)
payload_text += numeric_summary.to_string()
payload_text += '\n\n'

# 4. Missing values
payload_text += '=== MISSING VALUES ===\n'
missing_table = pd.DataFrame()
missing_table['missing_count'] = df.isna().sum()
missing_table['missing_pct'] = (df.isna().sum() / len(df) * 100).round(2)
payload_text += missing_table.to_string()
payload_text += '\n\n'

# 5. Unique values per column
payload_text += '=== UNIQUE VALUES PER COLUMN ===\n'
unique_table = pd.DataFrame()
unique_table['unique_count'] = df.nunique(dropna=False)
payload_text += unique_table.to_string()
payload_text += '\n\n'

# 6. Correlation between numeric columns
payload_text += '=== CORRELATION BETWEEN NUMERIC COLUMNS ===\n'
correlation_table = df.corr(numeric_only=True).round(2)
payload_text += correlation_table.to_string()
payload_text += '\n\n'

# 7. Top 10 values for categorical columns
payload_text += '=== TOP 10 VALUES FOR CATEGORICAL COLUMNS ===\n'

categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_columns) == 0:
    payload_text += 'No categorical columns found.\n'
else:
    for col in categorical_columns:
        payload_text += '\nColumn: ' + col + '\n'
        payload_text += df[col].value_counts(dropna=False).head(10).to_string()
        payload_text += '\n'

payload_text += '\n'

# 8. Simple warning checks
payload_text += '=== SIMPLE WARNING CHECKS ===\n'

id_like_columns = []
constant_columns = []

for col in df.columns:
    unique_count = df[col].nunique(dropna=False)

    if unique_count == len(df):
        id_like_columns.append(col)

    if unique_count <= 1:
        constant_columns.append(col)

payload_text += 'Possible ID-like columns: ' + str(id_like_columns) + '\n'
payload_text += 'Constant columns: ' + str(constant_columns) + '\n'
payload_text += 'Duplicate rows: ' + str(df.duplicated().sum()) + '\n'

print(payload_text)

# This cell builds a short dataset profile called payload_text.
# Instead of sending the full dataset to the LLM, we send summary information only.
# The payload includes shape, column types, numeric summary, missing values, unique counts, correlations, and common category values.
# It also adds simple warning checks for possible ID-like columns, constant columns, and duplicate rows.
# This helps the LLM comment on data readiness without needing every row of the dataset.

=== SHAPE ===
Rows: 395
Columns: 33

=== COLUMNS AND DATA TYPES ===
school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
G1             int64
G2             int64
G3             int64

=== NUMERIC SUMMARY ===
          age    Medu    Fedu  traveltime  studytime  failures  famrel  freetime   goout    Dalc    Walc  health  absences      G1      G2      G3
count  395.00  395.00  395.00      395.00     395.00    395.00  395

In [26]:
# Tum payload template  ******* Use this **********************
from io import StringIO

# ==========================================
# CHANGE ONLY THIS LINE
# ==========================================

target_column = None

# Example when target is known:
# target_column = "Price"

# ==========================================

buffer = StringIO()

# DATA TYPES
buffer.write("=== DATA TYPES ===\n")
buffer.write(df.dtypes.to_string())
buffer.write("\n\n")

# SHAPE
buffer.write("=== DATASET SHAPE ===\n")
buffer.write(f"Rows: {df.shape[0]}\n")
buffer.write(f"Columns: {df.shape[1]}\n\n")

# TARGET
buffer.write("=== TARGET COLUMN ===\n")

if target_column is None:
    buffer.write("Target not yet specified.\n")
else:
    buffer.write(f"Target: {target_column}\n")

    if target_column in df.columns:
        buffer.write(f"Data Type: {df[target_column].dtype}\n")
        buffer.write(
            f"Unique Values: {df[target_column].nunique(dropna=False)}\n"
        )

buffer.write("\n")

# MISSING VALUES
buffer.write("=== MISSING VALUES ===\n")

missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(
          missing_pct=lambda x:
          (x["missing_count"] / len(df) * 100).round(2)
      )
)

buffer.write(missing_summary.to_string())
buffer.write("\n\n")

# DUPLICATES
buffer.write("=== DUPLICATE ROWS ===\n")
buffer.write(str(df.duplicated().sum()))
buffer.write("\n\n")

# NUMERIC SUMMARY
buffer.write("=== NUMERIC SUMMARY ===\n")

try:
    buffer.write(
        df.describe(include="number")
          .round(3)
          .to_string()
    )
except:
    buffer.write("No numeric columns.")

buffer.write("\n\n")

# CATEGORICAL SUMMARY
buffer.write("=== CATEGORICAL SUMMARY ===\n")

try:
    buffer.write(
        df.describe(include=["object","category"])
          .to_string()
    )
except:
    buffer.write("No categorical columns.")

buffer.write("\n\n")

# UNIQUE VALUES
buffer.write("=== UNIQUE VALUES PER COLUMN ===\n")

buffer.write(
    df.nunique(dropna=False)
      .to_frame("unique_count")
      .to_string()
)

buffer.write("\n\n")

# CONSTANT COLUMNS
buffer.write("=== CONSTANT COLUMNS ===\n")

constant_cols = list(
    df.columns[
        df.nunique(dropna=False) <= 1
    ]
)

buffer.write(str(constant_cols))
buffer.write("\n\n")

# POTENTIAL ID COLUMNS
buffer.write("=== POTENTIAL ID COLUMNS ===\n")

id_cols = list(
    df.columns[
        df.nunique(dropna=False) == len(df)
    ]
)

buffer.write(str(id_cols))
buffer.write("\n\n")

# CORRELATION
buffer.write("=== CORRELATION MATRIX ===\n")

try:

    corr_matrix = (
        df.corr(numeric_only=True)
          .round(3)
    )

    if len(corr_matrix) > 0:
        buffer.write(corr_matrix.to_string())
    else:
        buffer.write("No numeric columns available.")

except Exception as e:
    buffer.write(str(e))

buffer.write("\n\n")

# TOP CATEGORICAL VALUES
buffer.write("=== TOP VALUES FOR CATEGORICAL COLUMNS ===\n")

cat_cols = df.select_dtypes(
    include=["object","category"]
).columns

if len(cat_cols) == 0:
    buffer.write("No categorical columns found.\n")

else:

    for col in cat_cols:

        buffer.write(f"\nColumn: {col}\n")

        buffer.write(
            df[col]
              .value_counts(dropna=False)
              .head(10)
              .to_string()
        )

        buffer.write("\n")

buffer.write("\n")

# SAMPLE ROWS
buffer.write("=== SAMPLE ROWS ===\n")
buffer.write(df.head().to_string())

payload_text = buffer.getvalue()

print(payload_text)

=== DATA TYPES ===
school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
G1             int64
G2             int64
G3             int64

=== DATASET SHAPE ===
Rows: 395
Columns: 33

=== TARGET COLUMN ===
Target not yet specified.

=== MISSING VALUES ===
            missing_count  missing_pct
school                  0          0.0
sex                     0          0.0
age                     0          0.0
address       


Dataset profile (no charts yet)

**Why this matters:** In a practical test, you should be able to describe your dataset without relying on charts.

Focus on:
- which columns are numeric vs categorical,
- missing values (how many, where),
- target column (what type it is, what values it takes).

Write your observations as short bullet points in your report.


###Mock question (revision of steps you have already practised)

1. Create a text-based payload that clearly describes the dataset.
The description should summarise the dataset structure, key columns, size, and any notable data issues.

2. Then, send this payload to the OpenAI API together with:

>>clear context about the task, and
>>
>>a concise instruction telling the model what you want it to do with the dataset information.

The following block shows an example in image form for revision purposes.

# **Step3 (A): LLM Dataset Reasoning Prompt**

Can skip this step if no need.

In [27]:
# Prompt 1: LLM Dataset Reasoning Prompt

reasoning_prompt = f"""
You are a senior data scientist preparing a practical-test report.

Dataset Profile:

{payload_text}

Please provide concise reasoning suitable for a markdown report.

Answer the following:

1. What does this dataset appear to contain?

2. Explain the key columns in plain business language.

3. What is the most likely target variable?
   - Explain why.
   - State whether the target is numeric or categorical.

4. Is the target likely to be balanced or imbalanced?
   - Explain the possible impact on model evaluation.

5. Is this likely to be a Regression or Classification problem?
   - Explain why.

6. Which evaluation metric(s) would be appropriate?
   - Explain why.

7. Summarise:
   - Numeric columns
   - Categorical columns
   - Missing values
   - Potential data-quality issues

Requirements:

- Keep each answer between 2 and 5 lines.
- Use practical business language.
- Use markdown bullet points.
- Avoid technical jargon where possible.
- Be suitable for a practical-test submission.
"""

reasoning_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
    You are helping a student prepare concise markdown observations for a data analytics practical test.

    Keep the answers short, practical and easy for a lecturer to mark.

    Do not provide multiple alternatives.
    Choose the most likely answer.
    """,
    input=reasoning_prompt
)

print(reasoning_response.output_text)

- **1) What does this dataset appear to contain?**  
  - Student survey/administrative data about secondary school learners.  
  - Includes background (family, address, guardian), school context, study habits, lifestyle, attendance, and academic results.  
  - Likely used to predict **student performance**.

- **2) Explain key columns in plain business language**  
  - **Background:** `sex`, `age`, `address` (urban/rural), `famsize`, `Pstatus`, `guardian`, parents’ education (`Medu`, `Fedu`).  
  - **School/study behavior:** `studytime`, `schoolsup`, `famsup`, `paid`, `internet`, `nursery`, `higher`, `activities`.  
  - **Lifestyle/attendance & performance signals:** drinking (`Dalc`, `Walc`), time away (`goout`), well-being (`health`), and `absences`, plus grades (`G1`, `G2`, `G3`).

- **3) Most likely target variable**  
  - **Target: `G3` (final grade)**. It represents the final academic outcome and is the most direct “end result” column.  
  - It is **numeric** (integer grade), sho

In [8]:
# Prompt 2: LLM Dataset Reasoning Prompt (Can skip this step if no need.)

reasoning_prompt = f"""
You are a senior data scientist preparing a practical-test report.

Dataset Profile:

{dataset_profile}

Please provide concise reasoning suitable for a markdown report.

Answer the following:
1. How many rows and columns there are.
2. What the first few rows look like.
3. Which columns are numeric or categorical.
4. Whether any obvious data quality issue appears.
5. Which column are we trying to predict?
6. Which columns look numeric?
7. Which columns look categorical?
8. Do you see any columns that may be discounts, IDs, or repeated constants?

This is a quick orientation step, not a full analysis.
"""

reasoning_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
You are helping a student prepare concise markdown observations for a data analytics practical test.

Keep the answers short, practical and easy for a lecturer to mark.

Do not provide multiple alternatives.
Choose the most likely answer.
""",
    input=reasoning_prompt
)

print(reasoning_response.output_text)

### 1) Rows and columns
- **Rows:** 395  
- **Columns:** 33  

### 2) First few rows (what they look like)
- Dataset is a **mixed-type tabular student performance** dataset.
- Example columns shown: `school`, `sex`, `age`, `address`, family/guardian fields, school-related indicators (e.g., `studytime`, `internet`), and grades `G1`, `G2`, `G3`.
- Sample row 0 includes: `school=GP`, `sex=F`, `age=18`, `famsize=GT3`, `Medu=4`, `Fedu=4`, and finishes with grades like `G1=6, G2=5, G3=6`.

### 3) Numeric vs categorical columns
**Numeric (int64):**  
`age, Medu, Fedu, traveltime, studytime, failures, famrel, freetime, goout, Dalc, Walc, health, absences, G1, G2, G3`

**Categorical (object):**  
`school, sex, address, famsize, Pstatus, Mjob, Fjob, reason, guardian, schoolsup, famsup, paid, activities, nursery, higher, internet, romantic`

### 4) Obvious data quality issues
- **No missing values** (all columns show 0 missing).
- **No duplicate rows**.
- Main potential “issue” to watch later (no

# **Step 4: LLM assisted problem framing**

# **Find the Target column**

In [28]:
problem_framing_prompt = f"""

You are a senior business analytics consultant.

Using the dataset information below, translate the dataset into a clear business problem and modelling objective.

Dataset Information:

{payload_text}

Requirements:

1. Write the BUSINESS PROBLEM in simple business language.
   - Do not mention machine learning, algorithms, features or predictors.
   - Focus on the real-world problem the organisation wants to solve.

2. Write the MODELLING OBJECTIVE in technical language.
   - State exactly what the model should predict.
   - State whether it is Regression or Classification.
   - Choose ONE best answer only.

3. Identify:
   - Target variable
   - Evaluation metric & Reason behind
   - Main stakeholders
   - Business value
   - Key risks

Keep answers concise.

Use exactly this format:

Business Problem:
...

Modelling Objective:
...

Target:
...

Problem Type:
Regression / Classification

Metric:
...

Stakeholders:
...

Business Value:
...

Risks:
...
"""

problem_framing_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
    You are a senior business analytics consultant.

    Translate business problems into machine learning objectives.

    Keep answers concise and practical.
    """,
    input=problem_framing_prompt
)

print(problem_framing_response.output_text)

Business Problem:
A school board wants to identify which students are at higher risk of underperforming academically so it can prioritize academic support (tutoring, interventions, and mentoring) early in the term and reduce the number of students who fall below expected grade levels.

Modelling Objective:
Build a model to predict the student’s final achievement score (G3) for each enrolled student, using their prior term performance and background/behavioral information available at the time of prediction. The model should output a continuous estimate of G3.

Target:
G3 (final grade)

Problem Type:
Regression

Metric:
Mean Absolute Error (MAE) — it measures the average absolute deviation between predicted and actual final grades, is intuitive for school performance (grade-point units), and is more robust than RMSE to outliers.

Stakeholders:
Students and parents; teachers and academic support staff; school principals; education operations/program managers; school board decision-makers

# **Step 5: Create Context**

In [11]:
context = """

You are an expert data scientist specialising in machine learning preprocessing and tree-based models.

Only use evidence provided in the dataset profile.

Do not guess information that is not present.

Do not perform encoding, scaling, feature selection, train/test split, modelling or evaluation.

Focus only on data understanding, data quality assessment and data cleaning preparation.

"""


# **Step 6: Create Instruction**

In [12]:
instruction = """

TASK

Review the dataset profile and identify:

1. Data quality issues.
2. Missing value concerns.
3. Duplicate record concerns.
4. Potential outlier concerns.
5. ID-like columns.
6. Constant or near-constant columns.
7. Redundant or highly correlated columns.
8. Potential target leakage or label leakage risks.
9. Data type issues.
10. Feature engineering opportunities.
11. Columns that may be candidates for removal before modelling.

For every issue provide:

- Issue
- Evidence from dataset profile
- Why it matters
- ML impact
- Recommended action
- Priority (High / Medium / Low)

Important:

- Do not recommend dropping a column solely because it appears in warning checks.
- Explain why before recommending removal.
- If evidence is insufficient, state:
  'Cannot be confirmed from dataset profile alone.'

Finally provide:

A. Data Quality Review
B. Recommended Actions
C. Suggested Cleaning Steps (in order)

Keep the answer concise but complete.
"""



# **Step 7: Create Final Payload**

In [13]:
outcome = f"""
CONTEXT:
{context}

DATASET INFORMATION:
{payload_text}

TASK:
{instruction}
"""

print(outcome)


CONTEXT:


You are an expert data scientist specialising in machine learning preprocessing and tree-based models.

Only use evidence provided in the dataset profile.

Do not guess information that is not present.

Do not perform encoding, scaling, feature selection, train/test split, modelling or evaluation.

Focus only on data understanding, data quality assessment and data cleaning preparation.



DATASET INFORMATION:
=== SHAPE ===
Rows: 395
Columns: 33

=== COLUMNS AND DATA TYPES ===
school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
fre

# **Step 8: Send Payload to Open AI & OpenAI recommendation Output/Response**

In [14]:
response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
    You are a professional data analyst writing a high-quality business analytics report.
    """,
    input=outcome
)

print(response.output_text)

## A. Data Quality Review

### 1) Data quality issues
**Issue:** No missing values detected  
- **Evidence from dataset profile:** Missing values table shows **0 missing** for all 33 columns.  
- **Why it matters:** Model imputation is unnecessary; however, other quality issues (e.g., invalid ranges) could still exist (not indicated here).  
- **ML impact:** None from missingness; preprocessing can focus on validation/range checks.  
- **Recommended action:** Run **range/category validity checks** for ordinal-like integer columns (cannot be confirmed from profile alone).  
- **Priority:** Low

**Issue:** Potential invalid/atypical numeric magnitude for an ordinal-like feature  
- **Evidence from dataset profile:** `absences` has **max = 75**, `G1/G2/G3` min–max are **0–19/20** range-like values, and many integer predictors have small discrete supports (e.g., `Dalc`, `Walc` min–max 1–5).  
- **Why it matters:** If some columns are supposed to be ordinal (e.g., alcohol/health) or bounded

# **Template combine  step 5-8 (Create context + Instruction + Final Payload + send to Open AI & with Open AI recommendation output/Response)**

In [31]:
# Prompt : Target Known
# eg. target_column = "Price"

context = """
You are an expert data scientist with extensive knowledge of tree-based models.

Always justify recommendations using evidence from the dataset profile.

You are reviewing a dataset before machine learning modelling.

The goal is to identify data quality issues and provide preprocessing recommendations.

The target variable has been specified in the dataset profile.

Focus on identifying issues that may impact predictive modelling performance.

Do not perform encoding, scaling, feature selection, train/test splitting, or modelling.

Focus only on data quality assessment and data cleaning.
"""

instruction = """
Review the dataset profile and identify:

1. Data quality issues that should be resolved before modelling.
2. Missing value concerns.
3. Duplicate record concerns.
4. Potential outlier concerns.
5. ID-like columns.
6. Constant or near-constant columns.
7. Highly correlated or potentially redundant columns.
8. Potential target leakage issues.
9. Potential label leakage issues.
10. Feature engineering opportunities.
11. Columns that should be removed before modelling.

For each issue provide:

- Explanation
- Evidence from dataset profile
- ML impact
- Recommendation
- Priority (High / Medium / Low)

Important:

- Assess potential leakage only when supported by evidence from the dataset profile.
- Do NOT recommend dropping a column solely because it appears in warning checks.
- If evidence is insufficient, state:
  'Cannot be confirmed from dataset profile alone.'
- Do not speculate.

Then generate Python code to handle the identified issues.

Requirements:

1. Create one helper function for each issue identified.
2. Create a wrapper function that calls these helper functions using True/False parameters.
3. Provide a single-line example showing how to run the wrapper function.
4. Do NOT perform encoding, scaling, feature selection, or modelling.
5. Focus only on data cleaning and preparation.

Output format:

A. Data Quality Review
B. Recommended Actions
C. Python Cleaning Functions
D. Wrapper Function
E. Example Usage
"""

outcome = f"""
CONTEXT:
{context}

DATASET INFORMATION:
{payload_text}

TASK:
{instruction}
"""

response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
You are a professional data analyst writing a high-quality business analytics report.
""",
    input=outcome
)

print(response.output_text)

## A. Data Quality Review

> Note: The dataset profile does **not specify the target column**, so leakage checks related to the specific label cannot be fully confirmed.

### 1) Missing value concerns
- **Explanation**: No missing values reported in any column.
- **Evidence from dataset profile**: “MISSING VALUES: … 0 missing_count 0.0 missing_pct” for all 33 columns.
- **ML impact**: Likely minimal for imputation, but other data-quality issues (invalid values, outliers) can still exist.
- **Recommendation**: No imputation required based on the provided profile. Still validate for “hidden” missingness (e.g., empty strings in categoricals) and numeric sentinel values.
- **Priority**: Low

### 2) Duplicate record concerns
- **Explanation**: No duplicates reported.
- **Evidence**: “DUPLICATE ROWS: 0”
- **ML impact**: Minimal.
- **Recommendation**: Still consider near-duplicates only if business rules suggest them (not evidenced here).
- **Priority**: Low

### 3) Potential outlier concerns

In [30]:
# Prompt : Target Unknown
# target_column = None

context = """
You are an expert data scientist with extensive knowledge of tree-based models.

Always justify recommendations using evidence from the dataset profile.

You are reviewing a dataset before machine learning modelling.

The goal is to identify data quality issues and provide preprocessing recommendations.

The target variable has NOT been specified.

Do not assume a target variable.

Do not assess target leakage or label leakage because the target is unknown.

Focus only on data quality assessment, exploratory review, and data cleaning preparation.

Do not perform encoding, scaling, feature selection, train/test splitting, or modelling.
"""

instruction = """
Review the dataset profile and identify:

1. Data quality issues that should be resolved before modelling.
2. Missing value concerns.
3. Duplicate record concerns.
4. Potential outlier concerns.
5. ID-like columns.
6. Constant or near-constant columns.
7. Highly correlated or potentially redundant columns.
8. Data type issues.
9. Possible target column candidates (if obvious).
10. Feature engineering opportunities.
11. Columns that may require further investigation before modelling.

For each issue provide:

- Explanation
- Evidence from dataset profile
- ML impact
- Recommendation
- Priority (High / Medium / Low)

Important:

- Do NOT assume a target variable.
- Do NOT assess target leakage.
- Do NOT recommend dropping a column solely because it appears in warning checks.
- If evidence is insufficient, state:
  'Cannot be confirmed from dataset profile alone.'
- Do not speculate.

Then generate Python code to handle the identified issues.

Requirements:

1. Create one helper function for each issue identified.
2. Create a wrapper function that calls these helper functions using True/False parameters.
3. Provide a single-line example showing how to run the wrapper function.
4. Do NOT perform encoding, scaling, feature selection, or modelling.
5. Focus only on data cleaning and preparation.

Output format:

A. Data Quality Review
B. Recommended Actions
C. Python Cleaning Functions
D. Wrapper Function
E. Example Usage
"""

outcome = f"""
CONTEXT:
{context}

DATASET INFORMATION:
{payload_text}

TASK:
{instruction}
"""

response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
You are a professional data analyst writing a high-quality business analytics report.
""",
    input=outcome
)

print(response.output_text)

## A. Data Quality Review

> Dataset: 395 rows × 33 columns, no duplicate rows, no missing values reported in the profile.

### 1) Data quality issues that should be resolved before modelling

**Issue 1.1 — Unexpected numeric ranges/outliers (notably `absences`, `G2/G3`, and grades)**
- **Explanation:** Several numeric columns have ranges that may be inconsistent with expected ordinal scales. This dataset appears to contain integer-coded survey/academic features and grades. The presence of a large `absences` maximum (75) and grade maxima up to 20 suggest outliers or encoding/scale inconsistencies that should be validated.
- **Evidence from dataset profile:**
  - `absences`: min 0, **max 75**, mean 5.709 (high spread; std 8.003)
  - `G1`: min 3, max 19; `G2`: min 0, max 19; `G3`: min 0, max 20
  - Correlations among `G1/G2/G3` are very high (0.801–0.905), suggesting redundancy but also indicating grades likely share a common scale.
- **ML impact:** Tree models are robust to outliers, bu

Context + Instruction + payload + Response

# **To use in Practical test**:The dataset profile was sent to the OpenAI API together with context and instructions requesting data quality assessment and preprocessing recommendations.

A text-based dataset payload was created using the dataset structure, column information, data types, missing values, duplicate counts and sample records. The payload was combined with task context and instructions before being sent to the OpenAI API for data quality and preprocessing recommendations.

# **Step 9: Review OpenAI Recommendations**

If have practical test question. Otherwise, can skip

In [14]:
# Prompt 1: Review Open AI Recommendations
review_prompt = f"""
The following recommendations were generated from a dataset review:

{response.output_text}

Review each recommendation.

For each recommendation:

1. State:
   - Accept
   - Modify
   - Reject

2. Give a concise reason (1-2 sentences).

3. Focus on:
   - Data quality
   - Missing values
   - Duplicates
   - Outliers
   - Target leakage
   - Feature engineering
   - Machine learning best practices

Return the answer in this format:

Recommendation:
Decision:
Reason:

Keep the answer concise and practical.
"""

review_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="You are a senior data scientist.",
    input=review_prompt
)

print(review_response.output_text)

Recommendation: Normalize categorical string columns (strip whitespace, unify case).  
Decision: **Accept**  
Reason: Reduces category fragmentation from casing/spacing differences, which directly improves generalization and prevents unseen-category issues at inference.

Recommendation: Validate and optionally cap extreme outliers in `absences` using robust quantiles/IQR.  
Decision: **Accept**  
Reason: A heavy-tailed `absences` distribution (max far above typical values) can dominate tree splits and increase variance; robust capping improves stability without distorting central tendency.

Recommendation: Target-leakage prevention by dropping grade columns based on the chosen target.  
Decision: **Accept**  
Reason: If the target is a later grade (commonly `G3`), including earlier grades (`G1`, `G2`) leaks future information and can inflate offline metrics.

Recommendation: Run missing-value sentinel cleanup (empty strings / literal tokens).  
Decision: **Modify**  
Reason: While `nul

In [15]:
# Prompt 2: Review Open AI Recommendations

review_prompt = f"""
Below are recommendations generated from a dataset review.

{response.output_text}

For each recommendation:

1. State whether it should be Accepted, Modified, or Rejected.
2. Explain why.
3. Identify any risks or limitations.
"""

review_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="You are a senior data scientist.",
    input=review_prompt
)

print(review_response.output_text)

Below is a recommendation-by-recommendation decision: **Accepted / Modified / Rejected**, with rationale, and risks/limitations.

---

## A. Data Quality Review

### 1) Missing value concerns (Priority: Low)
**Decision: Accepted (as-is)**
1. **Why:** If the profile shows `null_count=0` across all columns, imputation is not necessary. Your only caution is about *non-null sentinels* (e.g., `"NA"`, `""`, `"null"`)—that’s a real-world issue that this recommendation correctly addresses.
2. **Risks / limitations:**  
   - If the dataset truly uses only real nulls (and no sentinels), the added runtime checks are redundant.  
   - If you only clean empty-string/type issues on `object` columns, you might miss numeric sentinels (e.g., `-1`) unless you also validate numeric domains.

---

### 2) Duplicate record concerns (Priority: Low)
**Decision: Accepted (as-is)**
1. **Why:** “0 exact duplicates” suggests low risk, and keeping a lightweight duplicate drop is safe. It won’t hurt model performan

In [16]:
# Prompt 3: Review Open AI Recommendations

review_prompt = f"""
The following recommendations were generated from a dataset review:

{response.output_text}

Please review each recommendation.

For each recommendation:

1. State whether it should be:
   - Accept
   - Modify
   - Reject

2. Explain the reason for the decision.

3. Consider:
   - Data quality
   - Target leakage
   - Missing values
   - Outliers
   - Feature engineering
   - Machine learning best practices

Present the answer in the following format:

Recommendation:
Decision:
Reason:
"""

review_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
    You are a senior data scientist reviewing recommendations made by a junior analyst.
    Evaluate each recommendation critically and justify your decisions.
    """,
    input=review_prompt
)

print(review_response.output_text)

Recommendation:
Normalize categorical string columns (strip whitespace, unify case).
Decision: **Accept**
Reason:  
- **Data quality:** This directly addresses a common hidden issue in “null_count=0” datasets—different representations of the same category (e.g., `"yes"`, `" yes"`, `"Yes"`), which can fragment categories.  
- **Missing values:** Doesn’t impute; it reduces downstream mismatch risk.  
- **ML best practices:** Especially important if you use one-hot encoding or target encoding—category fragmentation increases sparsity/variance and can harm generalization.  
- **Leakage:** No leakage introduced.  
- **Outliers:** Not relevant.  
- **Modification (minor):** In the provided code, converting to `str` before sentinel handling can turn real missing values into the literal strings `"nan"`/`"None"`. You *do* try to clean `"nan"/"None"` later, but I’d still adjust the order (handle NaNs before `astype(str)`) to be safer.

---

Recommendation:
Validate and optionally cap extreme out

In [17]:
# Prompt 4: Review Open AI Recommendations

review_prompt = f"""
The following recommendations were generated from a dataset review:

{response.output_text}

Review each recommendation and determine whether it should be Accepted, Modified, or Rejected.

Consider the following:

- Prevention of target leakage
- Effect of duplicates
- Treatment of missing values
- Whether outliers represent valid observations
- Whether feature engineering is necessary at this stage
- Whether any predictors should be removed before modelling

Provide clear reasoning for every decision.

Format:

Recommendation:
Decision:
Reason:
"""

review_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
    You are a senior data scientist reviewing recommendations made by a junior analyst.
    Evaluate each recommendation critically and justify your decisions.
    """,
    input=review_prompt
)

print(review_response.output_text)

### 1) Normalize categorical string columns (strip whitespace, unify case)
**Recommendation:** Accept  
**Decision:** **Accepted (with modification)**  
**Reason:**  
- **Missing values:** Not directly, but it reduces *false categories* and “missing-like” inconsistencies (e.g., `" NA"`, `"na "`).  
- **Effect on duplicates:** None; orthogonal.  
- **Model impact:** This is a common, high-value data quality fix for string categoricals—prevents category fragmentation and unseen-category issues.  
- **Whether feature engineering is necessary:** Not required; this is pure cleaning.  
- **Modification needed:** In the provided code, the function coerces `NaN` to the string `"nan"` (`astype(str)`), then tries to fix it with `isin(["nan", "None", ...])`. That’s workable but brittle. Prefer: only strip/lowercase non-null entries, e.g., operate via `.where(notna, ...)` or use `df[col] = df[col].str.strip()` without forcing to string first.  
- **Priority fit:** This is truly **high** because it

###Mock question (quality of understanding): Review against the need for each recommendation.

**Sample answer**
1. Ok to remove duplicate if any.
2. G1 G2 G3 will be removed later.
3. Combining or reduction of predictors is not needed at this stage, not an issue for XGB.
4. Dropping outlier is premature at this stage, especially when we are interested in more special cases. [For example, age has outliers, but as we can observe later, older students had higher chance to failing subjects. It is a valid signal, not a noise.]

**Check in with your tutor, to see if your understanding is aligned with good practices of data science.**

# **Step 10: Data Cleaning Prompt (Universal template)**
With python code to run later

In [39]:
# =====================================================
# STEP: Generate Data Cleaning Code
# Universal Cleaning Prompt
# =====================================================

cleaning_prompt = f"""
DATASET PROFILE:

{payload_text}

OPENAI RECOMMENDATIONS:

{response.output_text}

TASK:

Generate a practical data cleaning solution based on the dataset profile and recommendations.

First provide:

A. Cleaning Plan

Use a numbered list.

For each item provide:

Issue:
Action:
Reason:
Priority:

Keep explanations concise and practical.

Then provide:

B. Python Code

Requirements:

1. Generate ONE complete Python code block.
2. Assume the dataframe is called df.
3. Store the cleaned dataframe as df_clean.
4. Include comments explaining major cleaning steps.
5. Implement the recommended cleaning actions where appropriate.
6. Perform only data cleaning tasks.
7. Use sensible data science judgement when generating the code.
8. Ensure the code is executable in a single Jupyter/Colab notebook cell.
9. The final line should display df_clean.

Do NOT perform:

- Encoding
- Scaling
- Train/Test Split
- Feature Selection
- Modelling

Focus only on preparing a cleaned dataset ready for the next machine learning step.

Output format:

A. Cleaning Plan

B. Python Code
"""

cleaning_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
You are a senior data scientist.

Generate practical and accurate data-cleaning code.

Use the dataset profile and recommendations to decide what cleaning steps should actually be implemented.

Avoid unnecessary cleaning actions.

The cleaning plan should be easy for a student to read during a practical test.

The Python solution should be a single executable code block that can be copied into one notebook cell and run immediately.
""",
    input=cleaning_prompt
)

print(cleaning_response.output_text)

## A. Cleaning Plan

1) **Issue:** No missing values reported, but hidden “missing-like” categorical values (empty/whitespace) can exist.  
   **Action:** Trim whitespace for all categorical columns; convert empty/whitespace-only strings to real `NaN`; normalize casing to lower-case.  
   **Reason:** Prevents subtle category fragmentation (e.g., `"yes "` vs `"yes"`).  
   **Priority:** Medium

2) **Issue:** Numeric sanity/outliers risk: `absences` has a wide range (max 75) and grades may have inconsistent bounds (e.g., `G1` min 3 while `G2/G3` min 0).  
   **Action:**  
   - Coerce numeric columns to numeric (errors -> `NaN` only if something is truly malformed).  
   - For `absences`, set negative values to `NaN` and cap extreme high values using a high quantile (winsorize).  
   - Clip grade columns `G1/G2/G3` to `[0, 20]`.  
   **Reason:** Removes clearly invalid values and reduces leverage of extreme outliers while keeping most data intact.  
   **Priority:** Medium

3) **Issue:** 

# **Step 10-1 : Copy the python code output from the prompt to do the cleaning.**

In [40]:
import numpy as np
import pandas as pd

# -----------------------------
# Cleaning steps based on dataset profile + recommendations
# -----------------------------

# If you know the target column, set it here (e.g., "G3"). Otherwise leave as None.
target_col = None  # <-- change if needed (conditional leakage prevention)

df_clean = df.copy()

# 1) String hygiene for categorical columns
cat_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
for c in cat_cols:
    # Convert to string-like safely, then trim whitespace
    df_clean[c] = df_clean[c].astype("object")
    # Replace empty/whitespace-only with NaN
    df_clean[c] = df_clean[c].replace(r'^\s*$', np.nan, regex=True)
    # Normalize whitespace and casing
    df_clean[c] = df_clean[c].astype(str).str.strip().str.lower()
    df_clean[c] = df_clean[c].replace("nan", np.nan)

# 2) Numeric sanity checks & outlier handling
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

# Coerce numeric columns (keeps valid numerics; converts truly malformed entries to NaN)
for c in num_cols:
    df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")

# absences: no negatives; cap extreme upper tail (robust winsorization)
if "absences" in df_clean.columns:
    neg_mask = df_clean["absences"] < 0
    if neg_mask.any():
        df_clean.loc[neg_mask, "absences"] = np.nan

    # Cap high tail to reduce extreme leverage while retaining most values
    q = df_clean["absences"].quantile(0.995)
    if pd.notna(q):
        df_clean["absences"] = np.where(df_clean["absences"] > q, q, df_clean["absences"])

# Grades: clip to plausible [0, 20] range (conservative given profile max around 19-20)
for g in ["G1", "G2", "G3"]:
    if g in df_clean.columns:
        df_clean[g] = df_clean[g].clip(lower=0, upper=20)

# 3) Conditional leakage prevention if target is one of the grade columns
if target_col is not None and target_col in ["G1", "G2", "G3"]:
    other_grades = [g for g in ["G1", "G2", "G3"] if g != target_col and g in df_clean.columns]
    df_clean = df_clean.drop(columns=other_grades, errors="ignore")

# Final check: keep shape changes only from cleaning actions above
df_clean

# Display cleaned dataframe
df_clean

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,gp,f,18,u,gt3,a,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6.0,5,6,6
1,gp,f,17,u,gt3,t,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4.0,5,5,6
2,gp,f,15,u,le3,t,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10.0,7,8,10
3,gp,f,15,u,gt3,t,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2.0,15,14,15
4,gp,f,16,u,gt3,t,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4.0,6,10,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,ms,m,20,u,le3,a,2,2,services,services,course,other,1,2,2,no,yes,yes,no,yes,yes,no,no,5,5,4,4,5,4,11.0,9,9,9
391,ms,m,17,u,le3,t,3,1,services,services,course,mother,2,1,0,no,no,no,no,no,yes,yes,no,2,4,5,3,4,2,3.0,14,16,16
392,ms,m,21,r,gt3,t,1,1,other,other,course,other,1,1,3,no,no,no,no,no,yes,no,no,5,5,3,3,3,3,3.0,10,8,7
393,ms,m,18,r,le3,t,3,2,services,other,course,mother,3,1,0,no,no,no,no,no,yes,yes,no,4,4,1,3,4,5,0.0,11,12,10


# **Step 10-2: To check data after cleaning**

In [43]:
# What to check after cleaning:
# 1. Did the number of rows stay the same?
# 2. Did the discount columns disappear?
# 3. Are there missing values that still need attention?
# 4. Does Price_SGD still exist as the target column?

print('Original rows:', df.shape[0])
print('Cleaned rows:', df_clean.shape[0])

print('\nMissing values after cleaning:')
missing_after_cleaning = df_clean.isna().sum()
display(missing_after_cleaning.sort_values(ascending=False).head(10))

print('\nColumns after cleaning:')
print(df_clean.columns.tolist())


Original rows: 395
Cleaned rows: 395

Missing values after cleaning:


,0
school,0
sex,0
age,0
address,0
famsize,0
Pstatus,0
Medu,0
Fedu,0
Mjob,0
Fjob,0



Columns after cleaning:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


# **Save clean dataset as .csv file for next session**

In [46]:
df_clean.to_csv('cleaned_Predicting Number of Failed Subjects.csv', index=False)

print('Saved cleaned dataset as cleaned_Predicting Number of Failed Subjects.csv')
print('You can use this file in Session 2.')

#Task: Do you know where the csv file is?

Saved cleaned dataset as cleaned_Predicting Number of Failed Subjects.csv
You can use this file in Session 2.


# **Step 11: Create Target (Special Target Engineering)**

# **Check points:**

print(df.columns.tolist())

1. Is there a target column already?
2. Does the question specify a target?
3. Do I need to engineer a new target?
4. Could any columns cause target leakage?

**Open AI to help which target to used (With prompt)**


Scenario 1: Target Already exists
| Dataset | Target |
|----------|----------|
| House Prices | price |
| Customer Churn | churn |
| Employee Attrition | attrition |
| Insurance | charges |

Scenario 2: Instructor Tells you how to create target

Create a target indicating whether a customer is high risk.




# **Another Prompt for target recommendation**

In [44]:
# Open AI prompt for target recommendation

target_prompt = f"""
Review the dataset profile below.

DATASET PROFILE:

{payload_text}

Recommend a suitable target variable for machine learning.

Explain:

1. Which column(s) should be used.
2. Whether the problem is:
   - Regression
   - Classification
3. Whether target engineering is needed.
4. Any target leakage risks.
5. Provide Python code to create the target.

Keep the explanation concise.
"""

target_response = client.responses.create(
    model="gpt-5.4-nano",
    instructions="""
    You are a senior data scientist.
    """,
    input=target_prompt
)

print(target_response.output_text)

### Recommended target
Use **`G3`** as the target (final grade in Math, typically 0–20).

This keeps the task aligned with the dataset’s semantics: `G1` and `G2` are earlier grades, while `G3` is the outcome you ultimately care about.

---

## 1) Which column(s) should be used
- **Primary target:** `G3` (recommended)
- Optional (but avoid leakage for most models): `G1`, `G2`
  - If you want to *predict G3*, you generally keep **only `G3`** as target.
  - You can still use `G1`/`G2` as **features** if your real-world scenario “knows current/previous grades”. If not, exclude them.

**Suggested feature set for a leakage-minimized “student-time” setting:** all columns **except** `G3` (and consider also excluding `G1`, `G2` unless they are known at prediction time).

---

## 2) Whether the problem is regression or classification
- **Regression:** `G3` as a continuous numeric grade (most direct)
- **Classification (optional):** binarize or categorize `G3` (e.g., pass/fail), but only if you h

In [ ]:
# Does the dataset already contain a target column?
# Scenario 1 Target already exists. This case Target column = price
# ============================================================
# SCenario 1: CREATE TARGET
# ============================================================

target_col = "price"

y = df[target_col]

X = df.drop(columns=[target_col])

In [ ]:
# Need to create a new target as dataset doesn't contain target column.
# Scenario 2 Instructor tell you how to create the target eg. "num_failed_subjects"
# ============================================================
# SCenario 2: CREATE TARGET
# ============================================================

# Create target
df["num_failed_subjects"] = (
    (df[["G1", "G2", "G3"]] < 10)
    .sum(axis=1)
)

# Define target and predictors
y = df["num_failed_subjects"]

X = df.drop(columns=["num_failed_subjects"])

# Quick checks
print("Target shape:", y.shape)
print("Predictor shape:", X.shape)
print(y.value_counts().sort_index())

display(
    df[["G1", "G2", "G3", "num_failed_subjects"]].head()
)

# Or another example
#df["high_risk"] = (
#    (df["late_payments"] > 3)
#).astype(int)
#``

Target shape: (395,)
Predictor shape: (395, 33)
num_failed_subjects
0    221
1     30
2     44
3    100
Name: count, dtype: int64


,G1,G2,G3,num_failed_subjects
0,5,6,6,3
1,5,5,6,3
2,7,8,10,2
3,15,14,15,0
4,6,10,10,1


Target shape: (395,) => 395 rows & 1 targte variable (y)

Predictor shape: (395,33) => 395 students and 33 predictor columns (X)

0    221   => 220 students failed 0 subjects. Value_counts () help to answer whether target balance or imbalanced?

The target variable ranges from 0 to 3 failed subjects.

Most students have 0 failed subjects, while relatively fewer students have 2 or 3 failed subjects. This suggests some imbalance in the target distribution, which should be considered when evaluating model performance.


##Step 3) Create the target: number of failed subjects (special target)

Here you convert **three subject grades** into **one target**:

- A subject is considered **failed** when grade `< 10`.
- You count how many of the three subjects are failed.

### Common mistakes to avoid
- Using `<= 10` by accident (that changes the definition).
- Counting missing values as fails (check for missing grades first).
- Forgetting to confirm the **range** of the new target (it should usually be 0–3).



##Step 4) Visual check of subject grades (G1, G2, G3)

**Purpose:** quickly see the grade distribution (0–20) for each subject.

What you should notice:
- Are grades skewed towards high or low?
- Are there many values below 10 (fails)?
- Are there weird values outside 0–20 (data quality issue)?

If there are values outside the expected range, you should investigate before modelling.


In [ ]:
# ------------------------------------------------------------
# TARGET ENGINEERING (special case)
# Code for Step 3 and 4
# Target = count of failed subjects based on G1/G2/G3 < 10
# This is still REGRESSION (numeric target), but values are discrete.
# ------------------------------------------------------------

# --------------------------------------------
# Count number of subjects with score < 10
# from G1, G2, G3 (0 to 3)
# --------------------------------------------
grade_cols = ["G1", "G2", "G3"]

# True/False per subject, then sum across subjects (row-wise)
df["num_failed_subjects"] = (df[grade_cols] < 10).sum(axis=1)

# quick check
df[grade_cols + ["num_failed_subjects"]].head()
# --------------------------------------------
# Hist plots (3 by 1) for G1, G2, G3
# Use matplotlib only (no seaborn)
# --------------------------------------------
import matplotlib.pyplot as plt

fig, axs = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

axs[0].hist(df["G1"].dropna(), bins=range(0, 22), edgecolor="black")
axs[0].set_title("G1")

axs[1].hist(df["G2"].dropna(), bins=range(0, 22), edgecolor="black")
axs[1].set_title("G2")

axs[2].hist(df["G3"].dropna(), bins=range(0, 22), edgecolor="black")
axs[2].set_title("G3")

for ax in axs:
    ax.set_xlim(-0.5, 20.5)

plt.tight_layout()
plt.show()

# Quick sanity check you should do (in your own notes/report):
# - What are the unique values of the target?
# - What is the proportion of 0, 1, 2, 3?
# This helps you understand if the dataset is imbalanced by count.


##Step 5) Create the target: number of failed subjects (special target)

Here you convert **three subject grades** into **one target**:

- A subject is considered **failed** when grade `< 10`.
- You count how many of the three subjects are failed.

### Common mistakes to avoid
- Using `<= 10` by accident (that changes the definition).
- Counting missing values as fails (check for missing grades first).
- Forgetting to confirm the **range** of the new target (it should usually be 0–3).


In [ ]:
# ------------------------------------------------------------
# TARGET ENGINEERING (special case)
# Target = count of failed subjects based on G1/G2/G3 < 10
# This is still REGRESSION (numeric target), but values are discrete.
# ------------------------------------------------------------
# Optional: also plot the target distribution (0–3)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df["num_failed_subjects"].dropna(), bins=[-0.5, 0.5, 1.5, 2.5, 3.5], edgecolor="black")
plt.xticks([0, 1, 2, 3])
plt.title("Number of failed subjects (score < 10) across G1, G2, G3")
plt.xlabel("num_failed_subjects")
plt.ylabel("count")
plt.show()

# Quick sanity check you should do (in your own notes/report):
# - What are the unique values of the target?
# - What is the proportion of 0, 1, 2, 3?
# This helps you understand if the dataset is imbalanced by count.

###Useful code, for plotting predictors against categorical-type (including our count of subjects) prediction, to spot meaningful correlations and outliers.

In [ ]:
# ============================================================
# Jitter plot: NUMERIC features vs DISCRETE target (fast)
# x = target classes, y = numeric values (one row facet per feature)
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def jitter_numeric_fast(
    df: pd.DataFrame,
    target_col: str,
    num_cols: list[str] | None = None,
    order: list | None = None,
    max_features: int = 12,
    sample: int | None = 8000,
    max_classes: int = 12,
    class_order: str = "auto",  # "auto" | "sorted" | "freq"
    dropna_target: bool = True,
    dropna_feature: bool = True
):
    """
    Fast jitter plot for numeric features vs a discrete target.

    x = target classes
    y = numeric feature values (one facet per feature)

    Works for:
    - integer-coded targets (e.g. 0,1,2,3 or 1..5)
    - categorical/string targets (e.g. 'Low','Med','High')
    """

    if target_col not in df.columns:
        raise KeyError(f"target_col='{target_col}' not found in df")

    d = df.copy()

    # Drop missing target (usually safest)
    if dropna_target:
        d = d.dropna(subset=[target_col])

    # Auto-detect numeric columns (exclude target)
    if num_cols is None:
        num_cols = d.select_dtypes(include=[np.number]).columns.tolist()
        num_cols = [c for c in num_cols if c != target_col]

    # Keep only existing numeric columns
    num_cols = [c for c in num_cols if c in d.columns and c != target_col]
    if len(num_cols) == 0:
        raise ValueError("No numeric columns found to plot (after excluding target_col).")

    num_cols = num_cols[:max_features]

    # Decide x-axis order for target classes
    if order is None:
        y = d[target_col]

        # Ordered categorical: respect its category order (cap to max_classes)
        if isinstance(y.dtype, pd.CategoricalDtype) and y.dtype.ordered:
            order = list(y.cat.categories)
            if len(order) > max_classes:
                order = y.value_counts().head(max_classes).index.tolist()

        # Numeric target: prefer sorted unique (cap to max_classes)
        elif pd.api.types.is_numeric_dtype(y):
            uniq = pd.unique(y.dropna())
            order = sorted(uniq.tolist())
            if len(order) > max_classes:
                order = sorted(y.value_counts().head(max_classes).index.tolist())

        # String / object target: most frequent classes
        else:
            vc = y.astype("object").value_counts(dropna=False)
            order = vc.head(max_classes).index.tolist()
            if class_order == "sorted":
                order = sorted(order)
            elif class_order == "freq":
                order = order
            else:
                # auto: keep freq order
                order = order

    # Filter to chosen classes
    d = d[d[target_col].isin(order)].copy()

    # Sample once for speed
    if sample is not None and len(d) > sample:
        d = d.sample(sample, random_state=42)

    # Long form
    frames = []
    for c in num_cols:
        tmp = d[[target_col, c]].copy()
        if dropna_feature:
            tmp = tmp.dropna(subset=[c])
        tmp = tmp.rename(columns={c: "value"})
        tmp["feature"] = c
        frames.append(tmp[[target_col, "feature", "value"]])

    long_df = pd.concat(frames, ignore_index=True)

    # Force target order in seaborn
    long_df[target_col] = pd.Categorical(long_df[target_col], categories=order, ordered=True)

    g = sns.catplot(
        data=long_df,
        x=target_col,
        y="value",
        row="feature",
        kind="strip",
        order=order,
        jitter=True,
        height=2.2,
        aspect=3.2,
        sharey=False
    )

    g.set_axis_labels(f"{target_col} (classes: {order})", "Numeric value")
    g.set_titles("{row_name}")
    plt.tight_layout()
    plt.show()

    return order, num_cols


# Example:
order_used, num_cols_used = jitter_numeric_fast(df, target_col="num_failed_subjects", sample=15000)
print(order_used, num_cols_used)


In [ ]:
# ============================================================
# Jitter plot: CATEGORICAL features vs DISCRETE target (fast)
# x = target classes, y = category levels (one row facet per feature)
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def jitter_categorical_fast(
    df: pd.DataFrame,
    target_col: str,
    cat_cols: list[str] | None = None,
    order: list | None = None,
    max_features: int = 12,
    sample: int | None = 8000,
    max_classes: int = 12,
    class_order: str = "auto",      # "auto" | "sorted" | "freq"
    dropna_target: bool = True,
    top_k: int = 15,                # keep top categories per feature, rest -> "Other"
    dropna_feature: bool = False,   # if False, show missing as "(Missing)"
    missing_label: str = "(Missing)",
    show_other: bool = True
):
    """
    Fast jitter plot for categorical features vs a discrete target.

    x = target classes
    y = categorical feature levels (one facet per feature)

    Notes:
    - For each categorical feature, rare categories are collapsed into "Other" (top_k kept).
    - Missing can be shown explicitly as missing_label.
    """

    if target_col not in df.columns:
        raise KeyError(f"target_col='{target_col}' not found in df")

    d = df.copy()

    # Drop missing target (usually safest)
    if dropna_target:
        d = d.dropna(subset=[target_col])

    # Decide x-axis order for target classes
    if order is None:
        y = d[target_col]

        if isinstance(y.dtype, pd.CategoricalDtype) and y.dtype.ordered:
            order = list(y.cat.categories)
            if len(order) > max_classes:
                order = y.value_counts().head(max_classes).index.tolist()

        elif pd.api.types.is_numeric_dtype(y):
            uniq = pd.unique(y.dropna())
            order = sorted(uniq.tolist())
            if len(order) > max_classes:
                order = sorted(y.value_counts().head(max_classes).index.tolist())

        else:
            vc = y.astype("object").value_counts(dropna=False)
            order = vc.head(max_classes).index.tolist()
            if class_order == "sorted":
                order = sorted(order)
            elif class_order == "freq":
                order = order
            else:
                order = order

    # Filter to chosen classes
    d = d[d[target_col].isin(order)].copy()

    # Sample once for speed
    if sample is not None and len(d) > sample:
        d = d.sample(sample, random_state=42)

    # Auto-detect categorical columns
    if cat_cols is None:
        cat_cols = d.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
        cat_cols = [c for c in cat_cols if c != target_col]

    cat_cols = [c for c in cat_cols if c in d.columns and c != target_col]
    if len(cat_cols) == 0:
        raise ValueError("No categorical columns found to plot (after excluding target_col).")

    cat_cols = cat_cols[:max_features]

    # Build long form with per-feature top_k collapsing
    frames = []
    for c in cat_cols:
        tmp = d[[target_col, c]].copy()

        if dropna_feature:
            tmp = tmp.dropna(subset=[c])
            tmp[c] = tmp[c].astype("object")
        else:
            tmp[c] = tmp[c].astype("object").fillna(missing_label)

        # Collapse rare categories to Other
        vc = tmp[c].value_counts(dropna=False)
        keep = vc.head(top_k).index.tolist()

        if show_other:
            tmp[c] = tmp[c].where(tmp[c].isin(keep), other="Other")
            # Make a stable y-order: keep list + "Other" (if present)
            y_order = [k for k in keep if k in tmp[c].unique()]
            if "Other" in tmp[c].unique() and "Other" not in y_order:
                y_order.append("Other")
        else:
            tmp = tmp[tmp[c].isin(keep)].copy()
            y_order = [k for k in keep if k in tmp[c].unique()]

        tmp = tmp.rename(columns={c: "value"})
        tmp["feature"] = c
        tmp["_y_order"] = [y_order] * len(tmp)
        frames.append(tmp[[target_col, "feature", "value", "_y_order"]])

    long_df = pd.concat(frames, ignore_index=True)

    # Force target order in seaborn
    long_df[target_col] = pd.Categorical(long_df[target_col], categories=order, ordered=True)

    # Faceted strip plot. We cannot pass a different y-order per facet directly,
    # so we do a small loop to respect each feature's y-order.
    features = long_df["feature"].unique().tolist()
    n = len(features)

    fig, axes = plt.subplots(n, 1, figsize=(12, max(2.2 * n, 2.2)), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, feat in zip(axes, features):
        sub = long_df[long_df["feature"] == feat].copy()
        y_order = sub["_y_order"].iloc[0]

        sns.stripplot(
            data=sub,
            x=target_col,
            y="value",
            order=order,
            jitter=True,
            size=3,
            ax=ax
        )

        ax.set_title(feat)
        ax.set_ylabel("Category")
        ax.set_xlabel("")
        ax.set_yticks(range(len(y_order)))
        ax.set_yticklabels(y_order)

    axes[-1].set_xlabel(f"{target_col} (classes: {order})")
    plt.tight_layout()
    plt.show()

    return order, cat_cols


# Example:
order_used, cat_cols_used = jitter_categorical_fast(df, target_col="num_failed_subjects", sample=15000, top_k=12)
print(order_used, cat_cols_used)


### What to look for in quick checks

Standard checks you should be able to explain:
- `df.shape`: how many rows and columns (scale)
- `df.info()`: data types and missing values (readiness)
- `df.describe(include="all")`: typical values, outliers, rare categories (risk)

Common edge cases:
- `0` might mean a real value, or it might mean *missing coded as 0* (depends on the dataset).
- Some columns look numeric but are actually IDs or codes. Treat those carefully.


In [ ]:
# Class balance (important for evaluation choices)
target_col = "num_failed_subjects"
df[target_col].value_counts(normalize=True)

In [ ]:
X = df.drop(columns=[target_col, "G2", "G3", "G1"])
y = df[target_col]
X.shape, y.shape


##Step 6) Split into train and test sets

**Goal:** ensure you evaluate on unseen data.

Revision reminders:
- Use the same `random_state` for reproducibility.
- For regression, you usually do **not** stratify (unless you bucket the target).
- Keep the test set untouched until final evaluation.


In [ ]:
# ------------------------------------------------------------
# TRAIN / TEST SPLIT
# Keep the test set for final evaluation only.
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y  # important for imbalanced
)

In [ ]:
#value count for y_train and y_test
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


### Train/test split: what you are protecting against

**Standard definition:**  
A train/test split separates data you learn from (train) and data you hold back for final checking (test). This helps you estimate how well the model will work on new cases.

**Why `stratify=y` matters:**  
If classes are imbalanced, stratification keeps the class ratios similar in train and test.

**Edge case:**  
If a class is extremely rare, stratified splitting can fail or create tiny class counts. In that case, you may need a different split strategy or simpler target grouping.



##Step 7) Preprocessing (critical for mixed columns)

Most real datasets contain both:
- numeric columns (need scaling sometimes, missing values handling)
- categorical columns (need encoding, missing values handling)

**Tip:** In a practical test, a clean `ColumnTransformer` + `Pipeline` is often the best answer because:
- it reduces leakage,
- it ensures the same steps apply to train and test,
- it makes your workflow reproducible.


In [ ]:
# ------------------------------------------------------------
# PREPROCESSOR
# Make sure preprocessing is inside a Pipeline to avoid leakage.
# ------------------------------------------------------------

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Identify columns by dtype
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ],
    remainder="drop"
)

num_features[:10], cat_features[:10]



##Step 8) Build a model pipeline

**Revision goal:** You should be able to explain:
- What the preprocessor does
- What the model does
- Why a pipeline prevents mistakes

### Special note for this target
Even though the true target is a small set of integers, your regression model might output non-integers.
That is normal.

If the task requires integer predictions, you can *post-process* predictions (rounding),
but only do this if instructed, and always state the trade-off (it can change metrics).

### Evaluate using regression metrics

In a practical test, explain metrics in simple terms:

- **MAE**: average absolute error (easy to explain in grade points)
- **RMSE**: penalises large errors more strongly
- **R²**: how much variance is explained (can be misleading if target range is small)

### Edge case reminder
Because your target range is small (often 0–3), R² may look low even if errors are small.
So you should always report MAE or RMSE too.


In [ ]:
# ------------------------------------`
# 0. NOTE: This block takes quite a while to run, do it before moving onto explanation of code
# ------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import ShuffleSplit
cv = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
# -------------------------------------------
# 1. Create pipelines for both models
# -------------------------------------------

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42))
])

pipe_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        random_state=42))
])

# -------------------------------------------
# 2. Define parameter grids
# Keep them small for speed and simplicity
# -------------------------------------------

param_grid_rf = {
  "regressor__n_estimators": [50, 200],
  'regressor__max_depth':[5,10, None],
  'regressor__criterion':['squared_error', 'absolute_error']
}

param_grid_xgb = {
    "regressor__n_estimators": [50, 200],
  'regressor__max_depth':[2,4,6],
  'regressor__eval_metric':['rmse', 'mae']
}

# -------------------------------------------
# 3. Create GridSearchCV objects
# -------------------------------------------

gs_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

gs_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

# -------------------------------------------
# 4. Fit both models
# (Students can run one at a time if needed)
# -------------------------------------------e

gs_rf.fit(X_train, y_train)
print("Random Forest grid search complete.")

gs_xgb.fit(X_train, y_train)
print("XGBoost grid search complete.")


In [ ]:
#setup a results df to hold training and test scores
#at this point, you should already decide what scoring metric you going to use
#regression: r2, mae, rmse, mape

#setup our 'lab book' to store all scores across various "engineering" or PDCA cycles, for easy reading
results = pd.DataFrame(['cv_mae_val', 'cv_std_val', 'cv_mae_train', 'cv_std_train','holdout_mae','best para'])

In [ ]:
import numpy as np
from sklearn.model_selection import ShuffleSplit, cross_validate

cv = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)

# -----------------------------------------
# 1) CV diagnostics on TRAIN only
#    cross_validate gives train_score and test_score
# -----------------------------------------
best_model = gs_xgb.best_estimator_

cv_out = cross_validate(
    best_model,
    X_train, y_train,
    cv=cv,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
    n_jobs=-1,
    error_score="raise"
)

# Convert negative MAE to positive MAE
val_mae_scores = -cv_out["test_score"]
train_mae_scores = -cv_out["train_score"]

cv_mae_val_mean = float(val_mae_scores.mean())
cv_mae_val_std  = float(val_mae_scores.std())

cv_mae_train_mean = float(train_mae_scores.mean())
cv_mae_train_std  = float(train_mae_scores.std())

# -----------------------------------------
# 2) Holdout MAE (fit once, evaluate once)
# -----------------------------------------
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
holdout_mae = float(np.mean(np.abs(y_test - y_pred)))

# -----------------------------------------
# 3) Save into results (same 6-row lab book)
# -----------------------------------------
results["XGB_llm_encode_tuned"] = [
    cv_mae_val_mean,
    cv_mae_val_std,
    cv_mae_train_mean,
    cv_mae_train_std,
    holdout_mae,
    gs_xgb.best_params_,
]

display(results)


### Grid search and CV: what to do if time is tight

**Standard definition:**  
Cross-validation (CV) repeats training on different splits to reduce “lucky/unlucky split” effects. Grid search tries multiple hyperparameter settings.

**Practical-test approach (recommended):**
1. Build a baseline model and report results.
2. Improve **one thing only** (for example: class weighting, simpler preprocessing, or a small grid).

If you run the full grid search, note the runtime and what you would do to speed it up.



##Step 9) Error analysis (what to look for)

When you plot predictions or residuals, check:

- Do you systematically over-predict or under-predict?
- Are errors bigger for certain ranges (e.g., predicting 2 or 3 fails)?
- Do you see outliers that might come from data issues?

This is where you decide what to improve next.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------------------
# USER SETTINGS (edit these only)
# -----------------------------------------
sort_col = "Mjob"   # any column in X_test (e.g. "Brand", "GPU", "Screen_Size_inch")
step_n  = 1         # sample every n-th row after sorting

# -----------------------------------------
# 1. Sort by chosen column
# -----------------------------------------
sorted_idx = X_test[sort_col].sort_values().index

X_sorted = X_test.loc[sorted_idx]
y_sorted = y_test.loc[sorted_idx]

# 🔄 XGBoost prediction instead of RF
y_pred_sorted = gs_xgb.best_estimator_.predict(X_sorted)

# Combine into one DataFrame
df_plot = X_sorted.copy()
df_plot["Actual"] = y_sorted.values
df_plot["Predicted"] = y_pred_sorted

# -----------------------------------------
# 2. Apply sampling with iloc[::step_n]
# -----------------------------------------
df_plot = df_plot.iloc[::step_n].reset_index(drop=True)

group_series = df_plot[sort_col]
unique_groups = group_series.unique()

# colour palette based on unique groups
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_groups)))

# Find spans for shading
spans = []
start = 0
for i in range(1, len(group_series)):
    if group_series[i] != group_series[i - 1]:
        spans.append((start, i - 1, group_series[i - 1]))
        start = i
spans.append((start, len(group_series) - 1, group_series.iloc[-1]))

# -----------------------------------------
# 3. Plot with shading + sampled lines
# -----------------------------------------
fig, ax = plt.subplots(figsize=(24, 6))

# Background shading (color spelling is correct)
for idx, (s, e, group_name) in enumerate(spans):
    ax.axvspan(s, e, color=colors[idx % len(colors)], alpha=0.15)

# Actual values
ax.plot(
    df_plot["Actual"].values,
    label=f"Actual (sorted by {sort_col}, sampled every {step_n})",
    color="blue"
)

# Predicted values (XGBoost)
ax.plot(
    df_plot["Predicted"].values,
    label="Predicted (sampled)",
    color="red"
)

ax.set_title(f"Actual vs Predicted Prices — XGBoost (Sorted by {sort_col})")
ax.set_xlabel(f"Sample Index (sorted by {sort_col}, sampled every {step_n})")
ax.set_ylabel("Price (SGD)")
ax.legend()

# Group labels
for idx, (s, e, group_name) in enumerate(spans):
    ax.text(
        (s + e) / 2,
        ax.get_ylim()[1] * 0.98,
        str(group_name),
        ha="center",
        va="top",
        fontsize=9
    )

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Helper: plot y_true sorted, with y_pred aligned to same order
# ------------------------------------------------------------
def plot_sorted_true_vs_pred(y_true, y_pred_dict, title, xlabel="Sorted sample index"):
    """
    y_true: 1D array-like
    y_pred_dict: dict of {label: y_pred (1D array-like)}
    """
    y_true = np.asarray(y_true).ravel()

    order = np.argsort(y_true)          # indices that would sort y_true
    y_true_sorted = y_true[order]

    plt.figure(figsize=(12, 5))
    plt.plot(y_true_sorted, label="y_true (sorted)")

    for label, y_pred in y_pred_dict.items():
        y_pred = np.asarray(y_pred).ravel()
        plt.plot(y_pred[order], label=f"{label} pred (aligned)")

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Target value")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 1) Get predictions for TRAIN and TEST for both models
# ------------------------------------------------------------
rf_train_pred  = gs_rf.best_estimator_.predict(X_train)
xgb_train_pred = gs_xgb.best_estimator_.predict(X_train)

rf_test_pred   = gs_rf.best_estimator_.predict(X_test)
xgb_test_pred  = gs_xgb.best_estimator_.predict(X_test)


# ------------------------------------------------------------
# 2) Plot TRAIN: y_train sorted + aligned preds
# ------------------------------------------------------------
plot_sorted_true_vs_pred(
    y_true=y_train,
    y_pred_dict={
        "RF": rf_train_pred,
        "XGB": xgb_train_pred
    },
    title="TRAIN: y_train (sorted) vs RF/XGB predictions (aligned to sorted order)"
)


# ------------------------------------------------------------
# 3) Plot TEST: y_test sorted + aligned preds
# ------------------------------------------------------------
plot_sorted_true_vs_pred(
    y_true=y_test,
    y_pred_dict={
        "RF": rf_test_pred,
        "XGB": xgb_test_pred
    },
    title="TEST: y_test (sorted) vs RF/XGB predictions (aligned to sorted order)"
)


### Early stopping (XGBoost): what it is and why it needs a validation set

**Standard definition:**  
Early stopping stops training when performance on a validation set stops improving, helping to reduce overfitting.

**Important caveat:**  
If you use early stopping, you need a validation set that is not used for fitting the final model at that moment. This is why the notebook creates `X_val, y_val`.

**Alternate interpretation:**  
Some workflows use CV with early stopping inside each fold, but that is more complex and slower. In a practical test, a single validation split is usually enough.



##Step 10) Model improvement (only if time allows)

In a practical test, you only do tuning if it is explicitly requested or you have enough time.

If you tune:
- keep the parameter grid small and sensible
- record best parameters
- re-evaluate on the test set once

**Do not** repeatedly test on the test set. That becomes leakage.


In [ ]:
# ------------------------------------------------------------
# TRAIN / TEST SPLIT
# Keep the test set for final evaluation only.
# ------------------------------------------------------------

%%time
# ============================
# XGBOOST REGRESSION TUNING (GridSearchCV with ShuffleSplit, MAE) + FINAL EARLY STOPPING (single block)
# ============================

import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, ShuffleSplit, train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from xgboost import XGBRegressor

# ----------------------------
# 0) Hold-out validation ONLY for final early stopping
# ----------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42
)

# ----------------------------
# 1) CV: ShuffleSplit (test_size=0.1)
# ----------------------------
cv = ShuffleSplit(n_splits=5, test_size=0.1, random_state=42)

# ----------------------------
# 2) Pipeline: smaller n_estimators for tuning speed
# ----------------------------
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        random_state=42,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        n_estimators=200,     # keep modest during GridSearch
        eval_metric="mae"
    ))
])

# ----------------------------
# 3) GridSearch: simple, sensible grid for "most hyperparameters"
#    (keep it small to avoid exploding runtime)
# ----------------------------
param_grid = {
    "regressor__learning_rate": [0.03, 0.05, 0.1],
    "regressor__max_depth": [2, 3, 5],
    "regressor__min_child_weight": [1, 5],
    "regressor__subsample": [0.8, 1.0],
    "regressor__colsample_bytree": [0.8, 1.0],
    "regressor__reg_lambda": [1.0, 5.0],
}

gs = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",  # MAE (negated because higher-is-better convention)
    cv=cv,
    refit=True,
    n_jobs=-1,
    verbose=1
)

# ----------------------------
# 4) Fit GridSearch on X_tr only
# ----------------------------
gs.fit(X_tr, y_tr)

print("\nGridSearch tuning complete.")
print("Best params:", gs.best_params_)
print("Best CV MAE:", -gs.best_score_)

# ----------------------------
# 5) Final refit with early stopping (based on X_val)
# ----------------------------
best_pipe = gs.best_estimator_
preproc = best_pipe.named_steps["preprocessor"]
best_xgb = best_pipe.named_steps["regressor"]

X_tr_p = preproc.fit_transform(X_tr)
X_val_p = preproc.transform(X_val)
X_test_p = preproc.transform(X_test)

final_params = best_xgb.get_params()
final_params["n_estimators"] = 5000
final_params["eval_metric"] = "mae"
final_params["early_stopping_rounds"] = 100

xgb_final = XGBRegressor(**final_params)
xgb_final.fit(
    X_tr_p, y_tr,
    eval_set=[(X_val_p, y_val)],
    verbose=False
)

# ----------------------------
# 6) Predict using best_iteration when available
# ----------------------------
best_iter = getattr(xgb_final, "best_iteration", None)

if best_iter is not None:
    y_pred = xgb_final.predict(X_test_p, iteration_range=(0, best_iter + 1))
else:
    # Fallback: use all boosted rounds (still OK, just not "best iteration" forced)
    y_pred = xgb_final.predict(X_test_p)

# Optional rounding if target is discrete (e.g. 0..3)
y_pred_round = np.clip(np.rint(y_pred), 0, 3)

print("\nEarly stopping check:")
print("best_iteration:", getattr(xgb_final, "best_iteration", None))
try:
    booster = xgb_final.get_booster()
    print("num_boosted_rounds:", booster.num_boosted_rounds())
except Exception as e:
    print("Could not read booster rounds:", repr(e))

print("\nTEST metrics (raw predictions):")
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))
print("R^2 :", r2_score(y_test, y_pred))

print("\nTEST metrics (rounded to 0..3, optional):")
print("MAE :", mean_absolute_error(y_test, y_pred_round))
print("RMSE:", root_mean_squared_error(y_test, y_pred_round))
print("R^2 :", r2_score(y_test, y_pred_round))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Helper: plot y_true sorted, with y_pred aligned to same order
# ------------------------------------------------------------
def plot_sorted_true_vs_pred(y_true, y_pred, title, xlabel="Sorted sample index"):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    order = np.argsort(y_true)
    y_true_sorted = y_true[order]

    plt.figure(figsize=(12, 5))
    plt.plot(y_true_sorted, label="y_true (sorted)")
    plt.plot(y_pred[order], label="best-fit prediction (aligned)")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Target value")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# ASSUMES you already ran the fast XGB block above and now have:
# - xgb_final (final early-stopped XGB)
# - preproc   (fitted preprocessor used for xgb_final)
# - X_train, X_test, y_train, y_test
# ------------------------------------------------------------

# Preprocess for xgb_final
X_train_p = preproc.transform(X_train)
X_test_p  = preproc.transform(X_test)

# Best-iteration prediction (version-safe)
best_iter = getattr(xgb_final, "best_iteration", None)
best_ntree_limit = getattr(xgb_final, "best_ntree_limit", None)

if best_iter is not None:
    train_pred = xgb_final.predict(X_train_p, iteration_range=(0, best_iter + 1))
    test_pred  = xgb_final.predict(X_test_p,  iteration_range=(0, best_iter + 1))
elif best_ntree_limit is not None:
    train_pred = xgb_final.predict(X_train_p, ntree_limit=best_ntree_limit)
    test_pred  = xgb_final.predict(X_test_p,  ntree_limit=best_ntree_limit)
else:
    train_pred = xgb_final.predict(X_train_p)
    test_pred  = xgb_final.predict(X_test_p)

# ------------------------------------------------------------
# Plot TRAIN and TEST using best-fit predictions only
# ------------------------------------------------------------
plot_sorted_true_vs_pred(
    y_true=y_train,
    y_pred=train_pred,
    title="TRAIN: y_train (sorted) vs XGB best-fit predictions (aligned)"
)

plot_sorted_true_vs_pred(
    y_true=y_test,
    y_pred=test_pred,
    title="TEST: y_test (sorted) vs XGB best-fit predictions (aligned)"
)